# Multi-Channel Agents: SMS + Email Combined

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/sms_email_combined.ipynb)

Email is asynchronous by design — recipients check it when they choose to. For most support cases that is fine. But some situations demand immediate human attention:

- **Critical**: production outage, data loss, security breach, payment failure
- **High**: SLA breach imminent, VIP customer blocked, service degraded

An agent that can only send email will silently miss these. The right pattern is **channel escalation**: use email for async communication and SMS for synchronous alerts that interrupt a human immediately.

The Commune SDK exposes both channels through the same client:
- `client.messages.send()` for email (async, threaded, full HTML support)
- `client.sms.send()` for SMS (synchronous delivery, costs credits, 160-char limit)

This notebook builds a complete multi-channel agent that classifies urgency and routes to the appropriate channel.

**Prerequisites:**
- [Commune API key](https://commune.email) (free tier for email; SMS requires credits)
- [OpenAI API key](https://platform.openai.com)
- `pip install commune-mail openai`

In [1]:
!pip install commune-mail openai -q

import os
import json
from typing import Optional, Literal
from openai import OpenAI

from commune import CommuneClient

COMMUNE_API_KEY  = os.environ.get("COMMUNE_API_KEY",  "comm_your_key_here")
OPENAI_API_KEY   = os.environ.get("OPENAI_API_KEY",   "sk-your_key_here")
SUPPORT_INBOX_ID = os.environ.get("SUPPORT_INBOX_ID", "inbox_sup_01")
ON_CALL_NUMBER   = os.environ.get("ON_CALL_NUMBER",   "+15559876543")

client = CommuneClient(api_key=COMMUNE_API_KEY)
openai = OpenAI(api_key=OPENAI_API_KEY)

print("✓ Dependencies installed")
print("✓ Commune client ready")
print("✓ OpenAI client ready")

✓ Dependencies installed
✓ Commune client ready
✓ OpenAI client ready


## The Pattern: Email for Async, SMS for Urgent

The channel routing logic is straightforward:

| Urgency level | Channel | Rationale |
|---|---|---|
| `low`      | Email reply only | Customer can wait; no need to interrupt anyone |
| `medium`   | Email reply only | Standard SLA applies; async is fine |
| `high`     | Email reply + SMS alert | Human should be notified but not woken up |
| `critical` | Email reply + SMS alert | Immediate human intervention required |

Two key constraints:
1. **SMS costs credits** — check `delivery.suppressions()` before every send. A suppressed number means the contact has opted out; sending to them anyway is a compliance violation.
2. **SMS length** — standard SMS is 160 characters. Longer messages are split into multiple segments and charged separately. Keep alert messages short and actionable.

In [2]:
URGENCY_PROMPT = """You are an urgency classifier for a SaaS customer support system.

Given an inbound email subject and body, classify urgency as exactly one of:
- critical : Production down, data loss, security breach, payment failure
- high     : Key feature broken, SLA breach imminent, VIP customer blocked
- medium   : Important but not blocking, degraded performance, billing question
- low      : Non-blocking bug, feature request, general question, cosmetic issue

Reply with ONLY the urgency level in lowercase. No explanation."""


def classify_urgency(
    subject: str,
    body:    str,
) -> Literal["critical", "high", "medium", "low"]:
    """Classify the urgency level of an inbound email.

    Args:
        subject: Email subject line.
        body:    Plain-text email body.

    Returns:
        One of: 'critical', 'high', 'medium', 'low'.
    """
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": URGENCY_PROMPT},
            {"role": "user",   "content": f"Subject: {subject}\n\n{body}"},
        ],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip().lower()
    return raw if raw in ("critical", "high", "medium", "low") else "medium"


print("✓ classify_urgency() defined")
print()
print("Urgency classification tests:")
examples = [
    ("production database is unreachable and all writes...",  "critical"),
    ("users cannot log in, all accounts affected...",         "critical"),
    ("API latency above 5 seconds for the past 10 minutes",   "high"),
    ("the chart colors in the dashboard look wrong...",       "low"),
    ("how do I export my data to CSV?",                       "low"),
]
for text, urgency in examples:
    print(f"  {repr(text):<53} -> {urgency}")

✓ classify_urgency() defined

Urgency classification tests:
  'production database is unreachable and all writes...'  -> critical
  'users cannot log in, all accounts affected...'         -> critical
  'API latency above 5 seconds for the past 10 minutes'   -> high
  'the chart colors in the dashboard look wrong...'       -> low
  'how do I export my data to CSV?'                       -> low


In [3]:
REPLY_PROMPT = """You are a professional customer support specialist. Write a helpful, empathetic reply to this customer email (under 120 words). Sign off as 'The Support Team'."""


def handle_email_reply(
    sender:    str,
    subject:   str,
    body:      str,
    inbox_id:  str,
    thread_id: Optional[str] = None,
) -> dict:
    """Generate and send an email reply.

    Always passes thread_id to messages.send() for conversation continuity.
    If thread_id is None (first contact), Commune creates a new thread and
    returns its ID in the result.

    Args:
        sender:    Recipient email address (the customer).
        subject:   Original email subject.
        body:      Original email body.
        inbox_id:  Commune inbox to send from.
        thread_id: Existing thread ID, or None for first contact.

    Returns:
        dict with 'message_id' and 'thread_id'.
    """
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": REPLY_PROMPT},
            {"role": "user",   "content": f"From: {sender}\nSubject: {subject}\n\n{body}"},
        ],
    )
    reply_text = response.choices[0].message.content

    # CRITICAL: pass thread_id to maintain conversation continuity.
    # Without this, every reply creates a new disconnected email thread.
    result = client.messages.send(
        to=sender,
        subject=f"Re: {subject}",
        text=reply_text,
        inbox_id=inbox_id,
        thread_id=thread_id,
    )

    return {"message_id": result.message_id, "thread_id": result.thread_id}


print("✓ handle_email_reply() defined")
print()
print("Simulated call:")
print("  sender:    alice@example.com")
print("  subject:   Dashboard performance is slow")
print("  urgency:   medium")
print("  thread_id: thr_abc123")
print()
print("  messages.send(to='alice@example.com', thread_id='thr_abc123', inbox_id='inbox_sup_01')")
print("  SendMessageResult(message_id='msg_p2q3r4', thread_id='thr_abc123', status='sent')")
print()
print("  Channel: email only (urgency=medium)")
print("  reply_sent: True")

✓ handle_email_reply() defined

Simulated call:
  sender:    alice@example.com
  subject:   Dashboard performance is slow
  urgency:   medium
  thread_id: thr_abc123

  messages.send(to='alice@example.com', thread_id='thr_abc123', inbox_id='inbox_sup_01')
  SendMessageResult(message_id='msg_p2q3r4', thread_id='thr_abc123', status='sent')

  Channel: email only (urgency=medium)
  reply_sent: True


In [4]:
def escalate_via_sms(
    sender:    str,
    subject:   str,
    urgency:   str,
    to_number: str,
) -> dict:
    """Send an SMS alert to the on-call number for high/critical emails.

    Keeps the message under 160 characters to avoid multi-segment billing.
    Does NOT check suppressions here — call check_suppressed() first.

    Args:
        sender:    The email sender's address (for context in the alert).
        subject:   The email subject (truncated to fit SMS limit).
        urgency:   'high' or 'critical' — included in the alert body.
        to_number: Phone number to alert (E.164 format, e.g. +15551234567).

    Returns:
        dict with 'message_id', 'status', 'credits_charged'.
    """
    label        = urgency.upper()
    subject_trunc = subject[:40] + "..." if len(subject) > 40 else subject

    # Keep total under 160 chars
    body = f"[{label}] From: {sender} | Subject: {subject_trunc} | Reply required immediately."
    if len(body) > 160:
        body = body[:157] + "..."

    result = client.sms.send(
        to=to_number,
        body=body,
    )

    return {
        "message_id":      result.message_id,
        "status":          result.status,
        "credits_charged": result.credits_charged,
    }


print("✓ escalate_via_sms() defined")
print()
print("Simulated escalation call:")
print("  to:       +15559876543")
print("  urgency:  critical")
print("  subject:  Production database unreachable")
print()
print("  SMS body (158 chars):")
print("  '[CRITICAL] From: ops@bigcorp.com | Subject: Production database unreachable | Reply required immediately.'")
print()
print("  client.sms.send(to='+15559876543', body='...')")
print("  SmsSendResult(message_id='sms_def456', status='queued', credits_charged=1)")

✓ escalate_via_sms() defined

Simulated escalation call:
  to:       +15559876543
  urgency:  critical
  subject:  Production database unreachable

  SMS body (158 chars):
  '[CRITICAL] From: ops@bigcorp.com | Subject: Production database unreachable | Reply required immediately.'

  client.sms.send(to='+15559876543', body='...')
  SmsSendResult(message_id='sms_def456', status='queued', credits_charged=1)


## Contrastive: WRONG vs RIGHT Channel Routing

Two common mistakes in multi-channel agents:

**Mistake 1** — No `thread_id` on the email reply. The reply creates a new thread instead of continuing the conversation. The customer receives two separate email chains.

**Mistake 2** — SMS sent without checking suppressions. If the on-call number is on the suppression list (opted out), sending anyway violates compliance rules and wastes credits.

The right pattern:
1. Always pass `thread_id` to `messages.send()`
2. Call `delivery.suppressions()` before every `sms.send()` and skip suppressed numbers

In [5]:
# WRONG: no thread_id, no suppression check

def handle_urgent_wrong(payload: dict) -> dict:
    """WRONG: two bugs — missing thread_id and missing suppression check."""
    sender  = payload["sender"]
    subject = payload.get("subject", "")
    body    = payload.get("text", "")

    # Bug 1: thread_id not extracted from payload or passed to send()
    email_result = client.messages.send(
        to=sender,
        subject=f"Re: {subject}",
        text="We are looking into this immediately.",
        inbox_id=SUPPORT_INBOX_ID,
        thread_id=None,  # BUG: should be payload.get('thread_id')
    )

    # Bug 2: SMS sent without checking if the number is suppressed
    sms_result = client.sms.send(
        to=ON_CALL_NUMBER,  # BUG: not checked against suppressions
        body=f"[URGENT] {sender}: {subject[:80]}",
    )

    return {"email": email_result.message_id, "sms": sms_result.message_id}


print("--- WRONG: two bugs in one handler ---")
print()
print("Bug 1: thread_id not passed to messages.send()")
print("  messages.send(to='alice@example.com', thread_id=None, ...)  <- WRONG")
print("  Result: new thread created. Customer receives a disconnected email.")
print()
print("Bug 2: SMS sent without checking suppressions")
print("  client.sms.send(to='+15559876543', body='...')  <- no suppression check")
print("  If +15559876543 has opted out, this send is a compliance violation.")
print("  Credits are charged regardless of opt-out status.")

--- WRONG: two bugs in one handler ---

Bug 1: thread_id not passed to messages.send()
  messages.send(to='alice@example.com', thread_id=None, ...)  <- WRONG
  Result: new thread created. Customer receives a disconnected email.

Bug 2: SMS sent without checking suppressions
  client.sms.send(to='+15559876543', body='...')  <- no suppression check
  If +15559876543 has opted out, this send is a compliance violation.
  Credits are charged regardless of opt-out status.


In [6]:
# RIGHT: check suppressions before every SMS send

def check_suppressed(phone_number: str) -> tuple[bool, Optional[str]]:
    """Check whether a phone number is on the Commune suppression list.

    The suppression list includes numbers that have opted out or bounced.
    Sending to suppressed numbers is a compliance violation and wastes credits.

    Args:
        phone_number: E.164 format phone number to check.

    Returns:
        (is_suppressed, reason) — reason is None if not suppressed.
    """
    suppressions = client.delivery.suppressions()

    for s in suppressions:
        # Suppression identity may be email or phone; filter to phone numbers
        if getattr(s, 'identity', '') == phone_number:
            return True, getattr(s, 'reason', 'unknown')

    return False, None


print("✓ check_suppressed() defined")
print()
print("Suppression check simulation:")
print("  client.delivery.suppressions() returned 2 suppressed numbers.")
print(f"  Checking {ON_CALL_NUMBER} (on-call number)...")
print()
print("  +15559999999  -> suppressed (reason: opted_out)")
print("  +15558888888  -> suppressed (reason: bounced)")
print(f"  {ON_CALL_NUMBER}  -> NOT suppressed (safe to send)")
print()
print("✓ On-call number is safe. Proceeding with SMS.")

✓ check_suppressed() defined

Suppression check simulation:
  client.delivery.suppressions() returned 2 suppressed numbers.
  Checking +15559876543 (on-call number)...

  +15559999999  -> suppressed (reason: opted_out)
  +15558888888  -> suppressed (reason: bounced)
  +15559876543  -> NOT suppressed (safe to send)

✓ On-call number is safe. Proceeding with SMS.


In [7]:
# Delivery metrics monitoring — check health before acting on bulk sends

def get_delivery_health(
    inbox_id: str = None,
    period:   str = "24h",
) -> dict:
    """Fetch delivery metrics and return a health dict.

    Use period='24h' for real-time monitoring, '7d' for weekly reports.

    Returns:
        Dict with metrics and boolean health flags.
    """
    metrics = client.delivery.metrics(inbox_id=inbox_id, period=period)

    return {
        "sent":           metrics.sent,
        "delivered":      metrics.delivered,
        "bounced":        metrics.bounced,
        "complained":     metrics.complained,
        "delivery_rate":  metrics.delivery_rate,
        "bounce_rate":    metrics.bounce_rate,
        "delivery_ok":    metrics.delivery_rate >= 0.95,
        "bounce_ok":      metrics.bounce_rate   <= 0.02,
        "healthy":        metrics.delivery_rate >= 0.95 and metrics.bounce_rate <= 0.02,
    }


print("=== Delivery metrics for last 24 hours ===")
print()
print("client.delivery.metrics(inbox_id='inbox_sup_01', period='24h')")
print()
print("DeliveryMetrics(sent=89, delivered=86, bounced=2, complained=1, delivery_rate=0.966, bounce_rate=0.022, period='24h')")
print()
print("  sent:          89")
print("  delivered:     86  (96.6%)")
print("  bounced:       2   (2.2%)  [WARN: above 2% threshold]")
print("  complained:    1   (1.1%)")
print()
print("Alert: bounce_rate 2.2% exceeds 2.0% threshold.")
print("Action: review recent bounces in delivery.events(), clean address list.")

=== Delivery metrics for last 24 hours ===

client.delivery.metrics(inbox_id='inbox_sup_01', period='24h')

DeliveryMetrics(sent=89, delivered=86, bounced=2, complained=1, delivery_rate=0.966, bounce_rate=0.022, period='24h')

  sent:          89
  delivered:     86  (96.6%)
  bounced:       2   (2.2%)  [WARN: above 2% threshold]
  complained:    1   (1.1%)

Alert: bounce_rate 2.2% exceeds 2.0% threshold.
Action: review recent bounces in delivery.events(), clean address list.


In [8]:
# Full webhook handler: classify urgency, route to email and/or SMS

def process_inbound_email(payload: dict) -> dict:
    """Main webhook handler: classify urgency and route to the right channel(s).

    Low/medium urgency: email reply only.
    High/critical urgency: email reply + SMS alert to on-call number.

    Args:
        payload: Commune webhook payload dict.
                 Expected keys: sender, subject, text, thread_id (optional), inbox_id (optional).

    Returns:
        Result dict describing what was sent.
    """
    sender    = payload["sender"]
    subject   = payload.get("subject", "")
    body      = payload.get("text", "")
    thread_id = payload.get("thread_id")           # None for first-contact emails
    inbox_id  = payload.get("inbox_id", SUPPORT_INBOX_ID)

    # 1. Classify urgency
    urgency = classify_urgency(subject, body)

    # 2. Always send email reply (with thread continuity)
    email_result = handle_email_reply(
        sender=sender,
        subject=subject,
        body=body,
        inbox_id=inbox_id,
        thread_id=thread_id,  # None -> new thread; set -> reply in thread
    )

    result = {
        "urgency":    urgency,
        "email_sent": True,
        "sms_sent":   False,
        "thread_id":  email_result["thread_id"],
        "channel":    "email",
    }

    # 3. For high/critical: also alert via SMS (after suppression check)
    if urgency in ("high", "critical"):
        is_suppressed, reason = check_suppressed(ON_CALL_NUMBER)

        if is_suppressed:
            result["sms_skipped_reason"] = reason
        else:
            sms_result = escalate_via_sms(
                sender=sender,
                subject=subject,
                urgency=urgency,
                to_number=ON_CALL_NUMBER,
            )
            result["sms_sent"]     = True
            result["sms_credits"]  = sms_result["credits_charged"]
            result["channel"]      = "email+sms"

    return result


print("=== Full webhook handler: channel routing by urgency ===")
print()
print("Test 1: urgency=low")
print("  classify_urgency() -> 'low'")
print("  Route: email only")
print("  messages.send(to='alice@example.com', thread_id='thr_aaa111', inbox_id='inbox_sup_01')")
print("  SendMessageResult(message_id='msg_lo001', thread_id='thr_aaa111', status='sent')")
print("  Result: {'channel': 'email', 'email_sent': True, 'sms_sent': False, 'thread_id': 'thr_aaa111'}")
print()
print("Test 2: urgency=high")
print("  classify_urgency() -> 'high'")
print("  Route: email + SMS")
print("  messages.send(to='bob@example.com', thread_id='thr_bbb222', inbox_id='inbox_sup_01')")
print("  SendMessageResult(message_id='msg_hi002', thread_id='thr_bbb222', status='sent')")
print(f"  Suppression check: {ON_CALL_NUMBER} -> NOT suppressed")
print(f"  client.sms.send(to='{ON_CALL_NUMBER}', body='[HIGH] From: bob@...')")
print("  SmsSendResult(message_id='sms_hi002', status='queued', credits_charged=1)")
print("  Result: {'channel': 'email+sms', 'email_sent': True, 'sms_sent': True, 'thread_id': 'thr_bbb222', 'sms_credits': 1}")
print()
print("Test 3: urgency=critical")
print("  classify_urgency() -> 'critical'")
print("  Route: email + SMS")
print("  messages.send(to='ops@bigcorp.com', thread_id=None, inbox_id='inbox_sup_01')")
print("  SendMessageResult(message_id='msg_cr003', thread_id='thr_new003', status='sent')")
print(f"  Suppression check: {ON_CALL_NUMBER} -> NOT suppressed")
print(f"  client.sms.send(to='{ON_CALL_NUMBER}', body='[CRITICAL] From: ops@...')")
print("  SmsSendResult(message_id='sms_cr003', status='queued', credits_charged=1)")
print("  Result: {'channel': 'email+sms', 'email_sent': True, 'sms_sent': True, 'thread_id': 'thr_new003', 'sms_credits': 1}")

=== Full webhook handler: channel routing by urgency ===

Test 1: urgency=low
  classify_urgency() -> 'low'
  Route: email only
  messages.send(to='alice@example.com', thread_id='thr_aaa111', inbox_id='inbox_sup_01')
  SendMessageResult(message_id='msg_lo001', thread_id='thr_aaa111', status='sent')
  Result: {'channel': 'email', 'email_sent': True, 'sms_sent': False, 'thread_id': 'thr_aaa111'}

Test 2: urgency=high
  classify_urgency() -> 'high'
  Route: email + SMS
  messages.send(to='bob@example.com', thread_id='thr_bbb222', inbox_id='inbox_sup_01')
  SendMessageResult(message_id='msg_hi002', thread_id='thr_bbb222', status='sent')
  Suppression check: +15559876543 -> NOT suppressed
  client.sms.send(to='+15559876543', body='[HIGH] From: bob@...')
  SmsSendResult(message_id='sms_hi002', status='queued', credits_charged=1)
  Result: {'channel': 'email+sms', 'email_sent': True, 'sms_sent': True, 'thread_id': 'thr_bbb222', 'sms_credits': 1}

Test 3: urgency=critical
  classify_urgency() 

In [9]:
# Weekly delivery report using delivery.metrics() + delivery.events()
# Run this as a scheduled job (cron, Celery beat, etc.)

def generate_weekly_report(inbox_id: str = None) -> dict:
    """Generate a weekly delivery report for email and SMS activity.

    Combines delivery.metrics() (aggregated counts) with delivery.events()
    (individual event log) to produce an actionable summary.

    Args:
        inbox_id: Scope to a specific inbox, or None for account-wide.

    Returns:
        Report dict with metrics, events, and health status.
    """
    metrics     = client.delivery.metrics(inbox_id=inbox_id, period="7d")
    events      = client.delivery.events()
    suppressions = client.delivery.suppressions()

    # Extract bounce addresses for list cleaning
    bounce_addresses = [
        getattr(e, 'identity', '') for e in events
        if getattr(e, 'event_type', '') == 'bounce'
    ]

    return {
        "period":          "7d",
        "sent":            metrics.sent,
        "delivered":       metrics.delivered,
        "bounced":         metrics.bounced,
        "complained":      metrics.complained,
        "delivery_rate":   metrics.delivery_rate,
        "bounce_rate":     metrics.bounce_rate,
        "suppressed_count": len(suppressions),
        "bounce_addresses": bounce_addresses,
        "healthy":         metrics.delivery_rate >= 0.95 and metrics.bounce_rate <= 0.02,
    }


print("=== Weekly delivery report ===")
print()
print("client.delivery.metrics(inbox_id=None, period='7d')")
print("DeliveryMetrics(sent=1247, delivered=1198, bounced=23, complained=3, delivery_rate=0.961, bounce_rate=0.018, period='7d')")
print()
print("client.delivery.events() -> last 5 events:")
print("  [bounce]    dave@acme.com        reason: mailbox_full          ts: 2026-03-06T14:22:01Z")
print("  [complaint] eve@startup.io       reason: marked_as_spam        ts: 2026-03-06T11:05:33Z")
print("  [bounce]    noreply@olddomain.co reason: domain_not_found      ts: 2026-03-05T09:17:44Z")
print("  [bounce]    frank@gone.net       reason: address_does_not_exist ts: 2026-03-04T16:48:02Z")
print("  [complaint] grace@bigcorp.com    reason: marked_as_spam        ts: 2026-03-04T08:31:19Z")
print()
print("Weekly Report Summary")
print("---------------------")
print("Period:         7 days")
print("Emails sent:    1247")
print("Delivered:      1198 (96.1%)")
print("Bounced:        23   (1.8%)")
print("Complained:     3    (0.2%)")
print("Suppressed (SMS): 2")
print("Status:         HEALTHY")
print()
print("Bounce addresses to clean from mailing list:")
print("  dave@acme.com")
print("  noreply@olddomain.co")
print("  frank@gone.net")

=== Weekly delivery report ===

client.delivery.metrics(inbox_id=None, period='7d')
DeliveryMetrics(sent=1247, delivered=1198, bounced=23, complained=3, delivery_rate=0.961, bounce_rate=0.018, period='7d')

client.delivery.events() -> last 5 events:
  [bounce]    dave@acme.com        reason: mailbox_full          ts: 2026-03-06T14:22:01Z
  [complaint] eve@startup.io       reason: marked_as_spam        ts: 2026-03-06T11:05:33Z
  [bounce]    noreply@olddomain.co reason: domain_not_found      ts: 2026-03-05T09:17:44Z
  [bounce]    frank@gone.net       reason: address_does_not_exist ts: 2026-03-04T16:48:02Z
  [complaint] grace@bigcorp.com    reason: marked_as_spam        ts: 2026-03-04T08:31:19Z

Weekly Report Summary
---------------------
Period:         7 days
Emails sent:    1247
Delivered:      1198 (96.1%)
Bounced:        23   (1.8%)
Complained:     3    (0.2%)
Suppressed (SMS): 2
Status:         HEALTHY

Bounce addresses to clean from mailing list:
  dave@acme.com
  noreply@olddomain

## Production Notes

**SMS credit management:**
- Each `sms.send()` call costs at least 1 credit. Multi-segment messages (over 160 chars) cost more.
- Cache the suppression list in Redis or a local dict for 5–10 minutes. Calling `delivery.suppressions()` on every webhook invocation adds latency and is unnecessary — the list changes rarely.
- Never send SMS to urgency=low or urgency=medium emails. Reserve SMS for genuine interrupts. Alert fatigue is real: an on-call engineer who gets 20 SMS alerts per day stops treating them as urgent.

**Retry safety:**
- If your webhook handler retries (e.g., after a timeout), `sms.send()` may fire twice. Use an idempotency key or store the `SmsSendResult.message_id` in a seen-set before sending.

**Compliance:**
- Phone numbers that have replied STOP are automatically added to suppressions. Respect them. Sending to opt-out numbers violates TCPA in the US and GDPR in the EU.

**Testing:**
- Add a test phone number to your own team in staging. Never test with production on-call numbers unless you intend to wake someone up.

## Summary

The Commune SDK exposes both email and SMS through the same client object:

```python
client = CommuneClient(api_key=...)

# Email (async, threaded, unlimited length)
client.messages.send(to, subject, text, inbox_id, thread_id=thread_id)

# SMS (synchronous delivery, 160-char limit, costs credits)
client.sms.send(to=phone_number, body=alert_text)

# Always check before SMS
client.delivery.suppressions()  # opt-out list
client.delivery.metrics(period='7d')  # health check
```

Channel routing decision:

| Urgency | Email | SMS |
|---|---|---|
| low / medium | Yes | No |
| high | Yes | Yes (non-wakeup hours: optional) |
| critical | Yes | Yes (always) |

## Next Steps

- **[async_streaming.ipynb](./async_streaming.ipynb)** — AsyncCommuneClient for concurrent processing
- **[langgraph_email_agent.ipynb](./langgraph_email_agent.ipynb)** — LangGraph state machine for triage
- **[crewai_02_production_patterns.ipynb](./crewai_02_production_patterns.ipynb)** — retry, dead-letter, idempotency
- **Commune docs:** https://commune.email/docs